# 4. Modeling: Pelatihan Model Tunggal LightGBM Classifier

Sesuai metodologi **CRISP-DM** (*Cross-Industry Standard Process for Data Mining*), tahap **Modeling** berfokus secara ketat pada:
1. Perancangan dan pemilihan arsitektur model pembelajaran mesin.
2. Pelatihan (*training / fitting*) model pada **data latih (*training set*)**.
3. Validasi performa internal menggunakan **Stratified 5-Fold Cross-Validation** pada data latih.
4. Serialisasi artefak model terlatih ke dalam direktori `models/`.

> **Catatan Metodologi CRISP-DM:**
> Seluruh pengujian pada data uji independen (*unseen test data*), pembuatan matriks konfusi (*confusion matrix*), kurva ROC-AUC, dan laporan metrik klasifikasi (Presisi, Recall, F1-Score) **sepenuhnya berada pada Tahap 5 (Evaluation)** guna mencegah kebocoran data (*data leakage*).


In [1]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('[OK] Library Pemodelan LightGBM berhasil dimuat.')


[OK] Library Pemodelan LightGBM berhasil dimuat.


## 4.1 Pemuatan Data Latih (Training Set) dan Konfigurasi Validasi Silang (Stratified 5-Fold CV)

Memuat **hanya data latih poligon (`train_data.csv`)** yang disiapkan dari Tahap 3. Fitur yang digunakan adalah **13 Fitur Spektral Pasti** (7 band BOA Sentinel-2 + 6 indeks biofisik teruji).


In [2]:
data_dir = os.path.join('..', '3-data_preparation', 'prepared_data')
train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))

# 13 Fitur Spektral Pasti (7 Band BOA + 6 Indeks Biofisik Teruji)
feature_cols = [
    'B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12',
    'NDVI', 'MNDWI', 'NDBI', 'Mangrove_Index_8A_11_4', 'EVI', 'BSI'
]

X_train = train_df[feature_cols].values
y_train = train_df['label'].values

class_names = ['Bangunan', 'Pertanian', 'Perairan', 'Hutan Daratan', 'Hutan Mangrove']

print(f'Dimensi Matriks Fitur Latih X : {X_train.shape[0]} sampel poligon x {X_train.shape[1]} fitur spektral')
print(f'Daftar 13 Fitur Pasti        : {feature_cols}')
print(f'Distribusi Sampel Latih per Kelas : {dict(pd.Series(y_train).value_counts().sort_index())}')

# Konfigurasi Stratified 5-Fold Cross Validation pada Data Latih
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('[OK] Stratified 5-Fold CV Generator siap digunakan.')


Dimensi Matriks Fitur Latih X : 173 sampel poligon x 13 fitur spektral
Daftar 13 Fitur Pasti        : ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'B12', 'NDVI', 'MNDWI', 'NDBI', 'Mangrove_Index_8A_11_4', 'EVI', 'BSI']
Distribusi Sampel Latih per Kelas : {0: np.int64(37), 1: np.int64(40), 2: np.int64(34), 3: np.int64(28), 4: np.int64(34)}
[OK] Stratified 5-Fold CV Generator siap digunakan.


## 4.2 Pelatihan Model LightGBM pada Data Latih Poligon

Model **LightGBM Classifier** dilatih menggunakan konfigurasi pohon *Gradient Boosting* yang optimal:
- `n_estimators=100`: Jumlah pohon pembelajar sekuensial yang cukup untuk konvergensi tanpa *overfitting*.
- `learning_rate=0.1`: Laju penyusutan gradien standar yang menjaga bobot residual tetap optimal.
- `num_leaves=31`: Kapasitas kompleksitas daun pohon yang seimbang.
- `subsample=1.0`: Menggunakan seluruh data latih di setiap pohon agar variasi sampel berukuran kecil tetap terlindungi penuh.


In [3]:
# Inisialisasi Model LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Validasi Silang (Stratified 5-Fold CV) pada Data Latih Saja
cv_scores = cross_val_score(lgb_model, X_train, y_train, cv=cv, scoring='accuracy')

# Pelatihan Model Penuh pada Seluruh Data Latih Poligon
lgb_model.fit(X_train, y_train)

print('HASIL VALIDASI SILANG DATA LATIH (STRATIFIED 5-FOLD CV):')
print(f'- Rata-rata Akurasi CV : {cv_scores.mean()*100:.2f}%')
print(f'- Standar Deviasi (Stabilitas) : ±{cv_scores.std()*100:.2f}%')
print(f'- Skor Tiap Fold : {[round(float(s)*100, 2) for s in cv_scores]}')
print('[OK] Pelatihan Model LightGBM Poligon selesai dengan sukses.')


HASIL VALIDASI SILANG DATA LATIH (STRATIFIED 5-FOLD CV):
- Rata-rata Akurasi CV : 79.19%
- Standar Deviasi (Stabilitas) : ±3.87%
- Skor Tiap Fold : [77.14, 80.0, 80.0, 85.29, 73.53]
[OK] Pelatihan Model LightGBM Poligon selesai dengan sukses.


## 4.3 Analisis Kepentingan Fitur Spektral (*Feature Importance*) Model LightGBM

Mengevaluasi kontribusi relatif masing-masing dari 13 fitur spektral pasti berdasarkan seberapa sering fitur tersebut digunakan untuk pemotongan pohon (*split importance*).


In [4]:
fi_vals = lgb_model.feature_importances_
fi_df = pd.DataFrame({
    'Fitur': feature_cols,
    'Importance': fi_vals,
    'Kontribusi (%)': (fi_vals / fi_vals.sum()) * 100
}).sort_values(by='Kontribusi (%)', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 5))
barplot = sns.barplot(data=fi_df, x='Kontribusi (%)', y='Fitur', palette='viridis')
for i, v in enumerate(fi_df['Kontribusi (%)']):
    barplot.text(v + 0.3, i, f'{v:.2f}%', va='center', fontweight='bold', fontsize=9)

plt.title('Peringkat Kepentingan 13 Fitur Spektral pada Model LightGBM', fontsize=13, pad=12)
plt.xlabel('Kontribusi Relatif (%)', fontsize=11)
plt.ylabel('Fitur Spektral', fontsize=11)
plt.xlim(0, fi_df['Kontribusi (%)'].max() + 3.0)
plt.tight_layout()
plt.show()
plt.close()

print('[OK] Tabel Kepentingan 13 Fitur Spektral:')
display(fi_df)


[OK] Tabel Kepentingan 13 Fitur Spektral:
                     Fitur  Importance  Kontribusi (%)
0                    MNDWI         496       12.963931
1   Mangrove_Index_8A_11_4         468       12.232096
2                     NDVI         370        9.670674
3                      B08         370        9.670674
4                      BSI         343        8.964976
5                     NDBI         326        8.520648
6                      B8A         272        7.109252
7                      B12         264        6.900157
8                      EVI         210        5.488761
9                      B11         193        5.044433
10                     B03         180        4.704652
11                     B02         176        4.600105
12                     B04         158        4.129639


## 4.4 Serialisasi Artefak Model Poligon Terlatih

Menyimpan model terlatih `model_lightgbm_teroptimasi.pkl` dan metadatanya `metadata_lightgbm_teroptimasi.json` ke folder `models/`.


In [5]:
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

poly_model_path = os.path.join(model_dir, 'model_lightgbm_teroptimasi.pkl')
poly_meta_path = os.path.join(model_dir, 'metadata_lightgbm_teroptimasi.json')

# Simpan Model Poligon
joblib.dump(lgb_model, poly_model_path)

# Definisi Metadata Terstruktur
classes_info = {
    '0': {'nama_kelas': 'Bangunan', 'alias': 'Built-up', 'kode_warna': '#EF4444'},
    '1': {'nama_kelas': 'Lahan Pertanian', 'alias': 'Farmland', 'kode_warna': '#D97706'},
    '2': {'nama_kelas': 'Perairan', 'alias': 'Water Body', 'kode_warna': '#0284C7'},
    '3': {'nama_kelas': 'Hutan Daratan', 'alias': 'Terrestrial Forest', 'kode_warna': '#16A34A'},
    '4': {'nama_kelas': 'Hutan Mangrove', 'alias': 'Mangrove Forest', 'kode_warna': '#0D9488'}
}

metadata = {
    'label_file': 'model_lightgbm_teroptimasi.pkl',
    'nama_model': 'LightGBM Classifier (Tuned Single Model)',
    'arsitektur': 'Histogram-Based Gradient Boosted Decision Tree (LightGBM)',
    'paradigma': '1 Model Tunggal Teroptimasi',
    'jumlah_fitur': len(feature_cols),
    'daftar_fitur': feature_cols,
    'kelas_target': classes_info,
    'n_sampel_latih': len(X_train),
    'rata_rata_akurasi_cv': float(cv_scores.mean()),
    'standar_deviasi_cv': float(cv_scores.std())
}

with open(poly_meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'[OK] Disimpan: model_lightgbm_teroptimasi.pkl ({os.path.getsize(poly_model_path)/1024:.1f} KB)')
print(f'[OK] Disimpan: metadata_lightgbm_teroptimasi.json ({os.path.getsize(poly_meta_path)/1024:.1f} KB)')


[OK] Disimpan: model_lightgbm_teroptimasi.pkl (535.7 KB)
[OK] Disimpan: metadata_lightgbm_teroptimasi.json (1.2 KB)


## 4.5 Pelatihan Model Spektral Piksel 10m Menggunakan LightGBM

Melatih model **LightGBM Tingkat Piksel** khusus pada dataset **`train_pixels.csv` (8.375 sampel latih murni)** untuk menghasilkan klasifikasi resolusi tinggi 10 meter yang presisi di setiap titik raster citra satelit Sentinel-2.


In [6]:
# Pemuatan Data Latih Piksel Bersih (Data Latih Saja)
train_pixels = pd.read_csv(os.path.join(data_dir, 'train_pixels.csv'))

X_tr_pix = train_pixels[feature_cols].values
y_tr_pix = train_pixels['label'].values

# Pelatihan Model Piksel LightGBM
lgb_pix = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.08,
    num_leaves=31,
    max_depth=8,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)
lgb_pix.fit(X_tr_pix, y_tr_pix)

# Simpan Model Piksel dan Metadata
pix_model_path = os.path.join(model_dir, 'model_lightgbm_piksel.pkl')
pix_meta_path = os.path.join(model_dir, 'metadata_lightgbm_piksel.json')

joblib.dump(lgb_pix, pix_model_path)

pix_meta = {
    'label_file': 'model_lightgbm_piksel.pkl',
    'nama_model': 'LightGBM Pixel Classifier (Tuned Single Model - 10m Resolution)',
    'paradigma': '1 Model Tunggal Teroptimasi Tingkat Piksel 10m',
    'daftar_fitur': feature_cols,
    'kelas_target': classes_info,
    'total_piksel_latih': len(X_tr_pix)
}
with open(pix_meta_path, 'w', encoding='utf-8') as f:
    json.dump(pix_meta, f, indent=2, ensure_ascii=False)

print(f'[OK] Disimpan: model_lightgbm_piksel.pkl ({os.path.getsize(pix_model_path)/1024:.1f} KB)')
print(f'[OK] Disimpan: metadata_lightgbm_piksel.json ({os.path.getsize(pix_meta_path)/1024:.1f} KB)')
print(f'[SELESAI] Model piksel berhasil dilatih pada {len(X_tr_pix):,} sampel data latih.')


[OK] Disimpan: model_lightgbm_piksel.pkl (3325.2 KB)
[OK] Disimpan: metadata_lightgbm_piksel.json (1.0 KB)
[SELESAI] Model piksel berhasil dilatih pada 8,375 sampel data latih.
